In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Simulation exacte avec les primitives du SDK Qiskit

<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
```
</details>
Les primitives de référence du SDK Qiskit effectuent des simulations locales par vecteur d'état. Ces simulations ne prennent pas en charge
la modélisation du bruit des appareils, mais elles sont utiles pour prototyper rapidement des algorithmes avant de se pencher sur des techniques de simulation plus avancées ([avec Qiskit Aer](./simulate-stabilizer-circuits)) ou d'exécuter sur de vrais appareils ([les primitives Qiskit Runtime](primitives)).

La primitive Estimator peut calculer des valeurs d'espérance de circuits, et la primitive Sampler peut échantillonner
à partir des distributions de sortie de circuits.

Les sections suivantes montrent comment utiliser les primitives de référence pour exécuter ton workflow localement.

## Utiliser l'Estimator de référence
L'implémentation de référence de `EstimatorV2` dans `qiskit.primitives` qui s'exécute sur des simulateurs locaux par vecteur d'état
est la classe [`StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator). Elle accepte des circuits, des observables et des paramètres en entrée, et retourne les valeurs d'espérance calculées localement.

Le code suivant prépare les entrées qui seront utilisées dans les exemples ci-dessous. Le type d'entrée attendu pour les
observables est [`qiskit.quantum_info.SparsePauliOp`](../api/qiskit/qiskit.quantum_info.SparsePauliOp). Note que
le circuit dans l'exemple est paramétré, mais tu peux aussi exécuter l'Estimator sur des circuits non paramétrés.

> **Note:** Tout circuit passé à un Estimator ne doit **pas** inclure de **mesures**.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# circuit for which you want to obtain the expected value
circuit = QuantumCircuit(2)
circuit.ry(Parameter("theta"), 0)
circuit.h(0)
circuit.cx(0, 1)
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/5b41a52d-8f15-4ce4-b3f6-effd91946d9c-0.svg" alt="Output of the previous code cell" />

In [2]:
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# observable(s) whose expected values you want to compute

observable = SparsePauliOp(["II", "XX", "YY", "ZZ"], coeffs=[1, 1, -1, 1])

# value(s) for the circuit parameter(s)
parameter_values = [[0], [np.pi / 6], [np.pi / 2]]

> **Tip:** Le workflow des primitives Qiskit Runtime requiert que les circuits et les observables soient transformés pour n'utiliser que les instructions prises en charge par le QPU (ce que l'on appelle des circuits et observables de *jeu d'instructions (ISA)*). Les primitives de référence acceptent toujours des instructions abstraites, car elles s'appuient sur des simulations locales par vecteur d'état, mais transpiler le circuit peut tout de même être bénéfique en termes d'optimisation.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(circuit)
>   isa_observable = observable.apply_layout(isa_circuit.layout)
>   ```

### Initialiser l'Estimator
Instancie un [`qiskit.primitives.StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator).

In [3]:
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

### Exécuter et obtenir les résultats
Cet exemple n'utilise qu'un seul circuit (de type [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit)) et un seul
observable.

Lance l'estimation en appelant la méthode [`StatevectorEstimator.run`](../api/qiskit/qiskit.primitives.StatevectorEstimator#run), qui retourne une instance d'un objet [`PrimitiveJob`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveJob). Tu peux obtenir les résultats du job (sous forme d'objet [`qiskit.primitives.PrimitiveResult`](../api/qiskit/qiskit.primitives.PrimitiveResult))
avec la méthode [`qiskit.primitives.PrimitiveJob.result`](../api/qiskit/qiskit.primitives.PrimitiveJob#result).

In [4]:
job = estimator.run([(circuit, observable, parameter_values)])
result = job.result()
print(f" > Result class: {type(result)}")

 > Result class: <class 'qiskit.primitives.containers.primitive_result.PrimitiveResult'>


#### Get the expected value from the result

The primitives result outputs an array of [`PubResult`](/docs/api/qiskit/qiskit.primitives.PubResult#pubresult) objects, where each item of the array is a `PubResult` object that contains in its data the array of evaluations corresponding to every circuit-observable combination in the PUB.

To retrieve the expectation values and metadata for the first (and in this case, only) circuit evaluation, we must access the evaluation [`data`](/docs/api/qiskit/qiskit.primitives.PubResult#data) for PUB 0:

In [5]:
print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

 > Expectation value: [4.         3.73205081 2.        ]
 > Metadata: {'target_precision': 0.0, 'circuit_metadata': {}}


#### Obtenir la valeur d'espérance depuis le résultat
Les résultats des primitives retournent un tableau d'objets [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult), où chaque élément du tableau est un objet `PubResult` qui contient dans ses données le tableau des évaluations correspondant à chaque combinaison circuit-observable dans le PUB.

Pour récupérer les valeurs d'espérance et les métadonnées pour la première (et dans ce cas, seule) évaluation de circuit, on doit accéder aux [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#data) d'évaluation du PUB 0 :

In [6]:
# Estimate expectation values for two PUBs, both with 0.05 precision.
precise_job = estimator.run(
    [(circuit, observable, parameter_values)], precision=0.05
)

For a full example, see the [Primitives examples](primitives-examples#estimator-examples) page.

## Use the reference Sampler

The reference implementations of `SamplerV2` in `qiskit.primitives` is the [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler) class. It takes circuits and parameters as inputs and returns the results from sampling from the output probability distributions as a quasi-probability distribution of output states.

The following code prepares the inputs used in the examples that follow. Note that
these examples run a single parametrized circuit, but you can also run the Sampler
on non-parametrized circuits.

In [7]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg" alt="Output of the previous code cell" />

### Configurer les options d'exécution de l'Estimator
Par défaut, l'Estimator de référence effectue un calcul exact par vecteur d'état en s'appuyant sur la classe
[`quantum_info.Statevector`](../api/qiskit/qiskit.quantum_info.Statevector).
Il est toutefois possible de le modifier pour introduire l'effet du surcoût d'échantillonnage (aussi appelé « bruit de grenaille »).

L'Estimator accepte un argument `precision` qui exprime les barres d'erreur que l'implémentation primitive doit cibler pour les estimations de valeurs d'espérance. Il s'agit du surcoût d'échantillonnage, défini exclusivement dans la méthode `.run()`. Cela te permet d'affiner l'option jusqu'au niveau PUB.

In [8]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

Pour un exemple complet, consulte la page [Exemples de primitives](primitives-examples#estimator-examples).

## Utiliser le Sampler de référence
L'implémentation de référence de `SamplerV2` dans `qiskit.primitives` est la classe [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler). Elle accepte des circuits et des paramètres en entrée, et retourne les résultats de l'échantillonnage des distributions de probabilité de sortie sous forme de distribution quasi-probabiliste des états de sortie.

Le code suivant prépare les entrées utilisées dans les exemples ci-dessous. Note que
ces exemples exécutent un seul circuit paramétré, mais tu peux aussi exécuter le Sampler
sur des circuits non paramétrés.

In [9]:
# execute 1 circuit with Sampler
job = sampler.run([circuit])
pub_result = job.result()[0]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg)

> **Note:** Tout circuit quantique passé à un Sampler **doit** inclure des mesures.

> **Tip:** Le workflow des primitives Qiskit Runtime requiert que les circuits soient transformés pour n'utiliser que les instructions prises en charge par le QPU (ce que l'on appelle des circuits ISA). Les primitives de référence acceptent toujours des instructions abstraites, car elles s'appuient sur des simulations locales par vecteur d'état, mais transpiler le circuit peut tout de même être bénéfique en termes d'optimisation.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(qc)
>   ```

### Initialiser `SamplerV2`
Instancie [`qiskit.primitives.StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler) :

In [10]:
from qiskit.transpiler import generate_preset_pass_manager

# create two circuits
circuit1 = circuit.copy()
circuit2 = circuit.copy()

# transpile circuits
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit1 = pm.run(circuit1)
isa_circuit2 = pm.run(circuit2)
# execute 2 circuits using Sampler
job = sampler.run([(isa_circuit1), (isa_circuit2)])
pub_result_1 = job.result()[0]
pub_result_2 = job.result()[1]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


### Exécuter et obtenir les résultats

In [11]:
# Define quantum circuit with 2 qubits
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw()

        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 

In [12]:
# Transpile circuit
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit = pm.run(circuit)
# Run using sampler
result = sampler.run([circuit]).result()
# Access result data for PUB 0
data_pub = result[0].data
# Access bitstring for the classical register "meas"
bitstrings = data_pub.meas.get_bitstrings()
print(f"The number of bitstrings is: {len(bitstrings)}")
# Get counts for the classical register "meas"
counts = data_pub.meas.get_counts()
print(f"The counts are: {counts}")

The number of bitstrings is: 1024
The counts are: {'11': 515, '00': 509}


Les primitives acceptent plusieurs PUBs en entrée, et chaque PUB obtient son propre résultat. Tu peux donc exécuter différents circuits avec diverses combinaisons de paramètres et d'observables, et récupérer les résultats des PUBs :

In [13]:
# Sample two circuits at 128 shots each.
sampler.run([isa_circuit1, isa_circuit2], shots=128)
# Sample two circuits at different amounts of shots. The "None"s are necessary
# as placeholders
# for the lack of parameter values in this example.
sampler.run([(isa_circuit1, None, 123), (isa_circuit2, None, 456)])

For a full example, see the [Primitives examples](./primitives-examples#sampler-examples) page.
## Next steps

<Admonition type="tip" title="Recommendations">
  - For higher-performance simulation that can handle larger circuits, or to incorporate noise models into your simulation, see [Exact and noisy simulation with Qiskit Aer primitives](simulate-with-qiskit-aer).
  - To learn how to use Quantum Composer for simulation, see the [IBM Quantum Composer](/docs/guides/composer) guide.
  - Read the [Qiskit Estimator API](/docs/api/qiskit/1.4/qiskit.primitives.Estimator) reference.
  - Read the [Qiskit Sampler API](/docs/api/qiskit/1.4/qiskit.primitives.Sampler) reference.
  - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>